# NAIP COG Builder (GDAL-First)

This notebook converts NAIP ZIP archives or TIFF imagery into **Cloud Optimized GeoTIFFs (COGs)** for two visualization modes and assembles aggregate VRTs across all images.

- **RGB** — bands `1,2,3` (natural color)
- **IRG** — bands `4,1,2` (infrared / CIR false color)

The implementation uses `osgeo.gdal` and `osgeo.osr` throughout.

In [28]:
# ── Environment preflight check ──────────────────────────────────────────────
# Run this cell FIRST to catch GDAL / Python-binding mismatches before they
# produce cryptic dlopen errors later in the workflow.
import subprocess, sys, importlib.util

def _check_gdal_env():
    issues = []
    gdal_ver = None

    # 1. Verify osgeo is importable at all.
    spec = importlib.util.find_spec('osgeo')
    if spec is None:
        issues.append('osgeo is not installed in this Python environment.')
    else:
        # Catch both ImportError (ModuleNotFoundError) and OSError (dlopen failure).
        try:
            from osgeo import gdal
            gdal_ver = gdal.__version__
        except (ImportError, OSError) as e:
            # ImportError / ModuleNotFoundError: osgeo package present but .so missing
            #   or a dependent .so (e.g. libgdal.38.dylib) cannot be found.
            # OSError: library loaded but something else went wrong at the C level.
            #
            # Most common cause: pip-installed GDAL bindings compiled against a
            # libgdal version absent from this env (e.g. libgdal.38 vs .37).
            #   Fix:  pip uninstall gdal -y
            #         conda install -n <env> -c conda-forge gdal=<version> -y
            #         (then restart the kernel)
            issues.append(
                f'osgeo import failed ({type(e).__name__}): {e}\n'
                '  Likely cause: pip-installed GDAL bindings compiled against a\n'
                '  libgdal version that does not exist in this environment.\n'
                '  Fix:\n'
                '    pip uninstall gdal -y\n'
                '    conda install -n <env> -c conda-forge gdal=<version> -y\n'
                '  Then restart the kernel.'
            )

    if gdal_ver:
        # 2. Require GDAL >= 3.4 for the OVERVIEWS=AUTO COG creation option.
        major, minor, *_ = (int(x) for x in gdal_ver.split('.'))
        if (major, minor) < (3, 4):
            issues.append(
                f'GDAL {gdal_ver} detected. OVERVIEWS=AUTO requires GDAL >= 3.4.'
            )
        else:
            print(f'GDAL version : {gdal_ver}  OK')

    # 3. Warn if a pip GDAL package is present alongside a conda one.
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', 'gdal'],
            capture_output=True, text=True, timeout=10,
        )
        if result.returncode == 0:
            pip_ver = next(
                (l.split(':', 1)[1].strip()
                 for l in result.stdout.splitlines()
                 if l.lower().startswith('version')),
                'unknown',
            )
            issues.append(
                f'pip-installed GDAL {pip_ver} is present in this environment.\n'
                '  This shadows conda-provided bindings and causes dlopen failures.\n'
                '  Fix:  pip uninstall gdal -y  (then restart the kernel)'
            )
    except Exception:
        pass

    if issues:
        print('=== GDAL environment issues ===')
        for i, msg in enumerate(issues, 1):
            print(f'[{i}] {msg}')
    else:
        print('Environment check passed.')

_check_gdal_env()

GDAL version : 3.11.4  OK
=== GDAL environment issues ===
[1] pip-installed GDAL 3.11.4 is present in this environment.
  This shadows conda-provided bindings and causes dlopen failures.
  Fix:  pip uninstall gdal -y  (then restart the kernel)


## What This Notebook Does

1. Scans one or more NAIP root directories recursively for ZIP archives and TIFF imagery.
2. If ZIPs are present, extracts them into local `prefire` or `postfire` folders based on the acquisition date encoded in the filename.
3. Treats acquisition year `2020` and earlier as `prefire`; later acquisition years are `postfire`.
4. For each discovered TIFF, calculates spatial metadata needed for COG creation.
5. Creates one canonical COG per source TIFF using GDAL, preserving all source bands.
6. Writes COGs to a local path based on the configured GeoJSON filename:
   - `./<geojson-stem>/cog/prefire/<image>.tif`
   - `./<geojson-stem>/cog/postfire/<image>.tif`

This keeps the source NAIP folders separate from the derived COG products.


## Cell 4: Configuration

Edit the values in the code cell below before running anything else.

| Variable | Purpose |
|---|---|
| `AOI_GEOJSON` | GeoJSON file used to name the local COG output folder. |
| `NAIP_ROOTS` | Local source paths to scan recursively for ZIPs and TIFFs. |
| `PREFIRE_YEAR` | Acquisition year treated as prefire; default is `2020`. |
| `OUTPUT_BASE_DIR` | Local output folder, normally `./<geojson-stem>`. |
| `COG_BLOCK_SIZE` | Internal tile size for COGs (512 is the de-facto standard). |
| `OVERVIEW_RESAMPLING` | Resampling method used when building overviews (`AVERAGE` for imagery). |
| `COG_COMPRESS` | Compression codec (`ZSTD` here for smaller lossless NAIP COGs). |
| `DRY_RUN` | `True` = report only; `False` = extract ZIPs and write COG/VRT files. |
| `OVERWRITE_OUTPUT` | Whether to overwrite existing COG output files. |
| `BUILD_VIZ_VRTS` | Build aggregate RGB/IRG VRTs from canonical 4-band COGs. |


In [29]:
from pathlib import Path
import math
import re
import shutil
import subprocess
import zipfile
from dataclasses import dataclass
from datetime import datetime
from typing import Iterable

from osgeo import gdal, osr

gdal.UseExceptions()

def first_existing_path(*candidates: str | Path) -> Path:
    # Return the first path that exists from a list of likely locations.
    # This makes the notebook work whether the kernel starts in the repo root
    # or in the notebooks/ folder.
    paths = [Path(candidate) for candidate in candidates]
    for path in paths:
        if path.exists():
            return path
    return paths[0]

# ----------------------------- User Configuration ----------------------------
# Example: wildfires/creek_2020.geojson -> ./creek_2020/naip/cog/prefire/
AOI_GEOJSON = first_existing_path(
    '../wildfires/north_complex_2020.geojson',
    './wildfires/north_complex_2020.geojson',
    './notebooks/wildfires/north_complex_2020.geojson',
)

# These folders are scanned recursively, so NAIP ZIPs may live beneath the configured root.
NAIP_ROOTS = [
    first_existing_path('./notebooks/downloads/north_complex_2020/naip/', './downloads/north_complex_2020/naip/'),
    # first_existing_path('./downloads/czu_aug_lightning_2020/naip', './notebooks/downloads/czu_aug_lightning_2020/naip'),
    # first_existing_path('./downloads/castle_2020/naip', './notebooks/downloads/castle_2020/naip'),
    # first_existing_path('./downloads/northcomplex/naip', './notebooks/downloads/northcomplex/naip'),
 ]

# Acquisition year used for prefire classification. NAIP filenames end with YYYYMMDD.
# With PREFIRE_YEAR = 2020, 2020 imagery is prefire and later imagery is postfire.
PREFIRE_YEAR = 2020

# Local COG output folder base. Output paths will be:
#    ./north_complex_2020/naip/cog/prefire/
#    ./north_complex_2020/naip/cog/postfire/
#    ./north_complex_2020/naip/vrt/
OUTPUT_BASE_DIR = Path('.') / 'north_complex_2020'
if AOI_GEOJSON.exists() and AOI_GEOJSON.is_file():
    OUTPUT_BASE_DIR = Path('.') / AOI_GEOJSON.stem

# Raster and ZIP extensions to include when scanning recursively.
IMAGE_EXTENSIONS = {'.tif', '.tiff'}
ZIP_EXTENSIONS = {'.zip'}

# Fallback EPSG if imagery lacks an embedded CRS (set to None to disable).
FALLBACK_EPSG = 26911

# Internal tile block size for COG output (512 px is the de-facto standard).
COG_BLOCK_SIZE = 512

# Resampling used when building overview levels.
# AVERAGE smooths well for continuous imagery; NEAREST preserves exact values.
OVERVIEW_RESAMPLING = 'AVERAGE'

# Compression settings tuned for smaller lossless COGs at full resolution.
# ZSTD is typically smaller than DEFLATE for NAIP-style imagery with no loss in detail.
COG_COMPRESS = 'ZSTD'
COG_LEVEL = 12
COG_PREDICTOR = 'STANDARD'
COG_NUM_THREADS = 'ALL_CPUS'

# Keep overviews lossless too so the whole COG stays analytically faithful.
OVERVIEW_COMPRESS = 'ZSTD'
OVERVIEW_PREDICTOR = 'STANDARD'

# Sub-folder under OUTPUT_BASE_DIR for COG output.
# COGs are written to ./<geojson-stem>/naip/cog/prefire or ./<geojson-stem>/naip/cog/postfire.
COGS_DIRNAME = 'cog'

# Optional viz-band aggregate VRT outputs built from canonical COGs.
BUILD_VIZ_VRTS = True
RGB_BANDS = (1, 2, 3)  # Natural color
IRG_BANDS = (4, 1, 2)  # Infrared, Red, Green (CIR false color)
AGGREGATE_DIR = OUTPUT_BASE_DIR / 'naip' / 'vrt'
RGB_AGGREGATE_VRT = AGGREGATE_DIR / 'rgb_all.vrt'
IRG_AGGREGATE_VRT = AGGREGATE_DIR / 'irg_all.vrt'

# Set DRY_RUN = False to actually extract ZIPs and write COG/VRT files.
DRY_RUN = False
OVERWRITE_OUTPUT = True  # Set to True to allow overwriting existing COGs and VRTs.

def final_cog_dir_for_phase(phase: str) -> Path:
    # Keep all generated COGs together under the GeoJSON-named output folder.
    return OUTPUT_BASE_DIR / 'naip' / COGS_DIRNAME / phase

# Print active configuration so path mistakes are easy to catch before processing.
print(f'AOI_GEOJSON:         {AOI_GEOJSON.resolve()}')
print(f'OUTPUT_BASE_DIR:     {OUTPUT_BASE_DIR.resolve()}')
print(f'PREFIRE_YEAR:        {PREFIRE_YEAR}')
print(f'NAIP_ROOTS ({len(NAIP_ROOTS)}):')
for root in NAIP_ROOTS:
    print(f'  {root.resolve()}')
print(f'Prefire COG dir:     {final_cog_dir_for_phase("prefire").resolve()}')
print(f'Postfire COG dir:    {final_cog_dir_for_phase("postfire").resolve()}')
print(f'COG_BLOCK_SIZE:      {COG_BLOCK_SIZE}')
print(f'OVERVIEW_RESAMPLING: {OVERVIEW_RESAMPLING}')
print(f'COG_COMPRESS:        {COG_COMPRESS}  (LEVEL={COG_LEVEL}, PREDICTOR={COG_PREDICTOR})')
print(f'OVERVIEW_COMPRESS:   {OVERVIEW_COMPRESS}  (PREDICTOR={OVERVIEW_PREDICTOR})')
print(f'COG_NUM_THREADS:     {COG_NUM_THREADS}')
print(f'BUILD_VIZ_VRTS:      {BUILD_VIZ_VRTS}')
print(f'AGGREGATE_DIR:       {AGGREGATE_DIR.resolve()}')
print(f'DRY_RUN:             {DRY_RUN}')
print(f'OVERWRITE_OUTPUT:    {OVERWRITE_OUTPUT}')


AOI_GEOJSON:         /Users/maples/GitHub/EarthExplorer_API/notebooks/notebooks/downloads/north_complex_2020/naip
OUTPUT_BASE_DIR:     /Users/maples/GitHub/EarthExplorer_API/notebooks/naip
PREFIRE_YEAR:        2020
NAIP_ROOTS (1):
  /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/north_complex_2020/naip
Prefire COG dir:     /Users/maples/GitHub/EarthExplorer_API/notebooks/naip/naip/cog/prefire
Postfire COG dir:    /Users/maples/GitHub/EarthExplorer_API/notebooks/naip/naip/cog/postfire
COG_BLOCK_SIZE:      512
OVERVIEW_RESAMPLING: AVERAGE
COG_COMPRESS:        ZSTD  (LEVEL=12, PREDICTOR=STANDARD)
OVERVIEW_COMPRESS:   ZSTD  (PREDICTOR=STANDARD)
COG_NUM_THREADS:     ALL_CPUS
BUILD_VIZ_VRTS:      True
AGGREGATE_DIR:       /Users/maples/GitHub/EarthExplorer_API/notebooks/naip/naip/vrt
DRY_RUN:             False
OVERWRITE_OUTPUT:    True


## Cell 6: Helper Functions (Part 1)

This first helper cell covers image discovery and spatial metadata math:

- `CogPlan` dataclass
- `iter_images`
- CRS helpers (`make_spatial_ref_from_epsg`, `get_dataset_srs`)
- GSD and zoom helpers (`estimate_gsd_meters`, `zoom_for_full_resolution`)
- overview calculator (`calculate_overview_levels`)

In [30]:
@dataclass
class CogPlan:
    # A single source image and its canonical COG output target.
    image_path: Path
    output_cog: Path
    size_bytes: int
    band_count: int
    width: int
    height: int
    gsd_m: float | None
    native_zoom: int | None
    overview_levels: list[int]
    source_is_cog: bool
    phase: str | None  # 'prefire', 'postfire', or None when unknown


def human_size(num_bytes: int) -> str:
    # Convert bytes into an easy-to-read text label.
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024
    return f'{num_bytes} B'


def should_skip_generated_path(path: Path, root_dir: Path) -> bool:
    # Avoid re-processing outputs if a broad NAIP root accidentally includes them.
    try:
        rel_parts = path.resolve().relative_to(root_dir.resolve()).parts
    except ValueError:
        return False
    generated_folder_names = {COGS_DIRNAME, AGGREGATE_DIR.name, OUTPUT_BASE_DIR.name}
    return any(part in generated_folder_names for part in rel_parts[:-1])


def iter_images(root_dir: Path) -> Iterable[Path]:
    # Recursively yield all source TIFF files under a root folder.
    root_dir = root_dir.resolve()
    if not root_dir.exists():
        return
    for path in root_dir.rglob('*'):
        if should_skip_generated_path(path, root_dir):
            continue
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            yield path


def iter_zips(root_dir: Path) -> Iterable[Path]:
    # Recursively yield ZIP files under a root folder.
    root_dir = root_dir.resolve()
    if not root_dir.exists():
        return
    for path in root_dir.rglob('*'):
        if should_skip_generated_path(path, root_dir):
            continue
        if path.is_file() and path.suffix.lower() in ZIP_EXTENSIONS:
            yield path


def parse_acquisition_date_from_name(path: Path) -> datetime:
    # NAIP filenames normally end with an acquisition date token: ..._YYYYMMDD.
    match = re.search(r'(\d{8})(?=$|\D)', path.stem)
    if not match:
        raise ValueError(f'Could not find YYYYMMDD acquisition date in {path.name}')
    return datetime.strptime(match.group(1), '%Y%m%d')


def classify_phase_from_acquisition(path: Path) -> str:
    # 2020 imagery is prefire for this workflow; imagery after 2020 is postfire.
    acquisition_date = parse_acquisition_date_from_name(path)
    return 'prefire' if acquisition_date.year <= PREFIRE_YEAR else 'postfire'


def infer_fire_phase(path: Path) -> str | None:
    # Prefer explicit folder names, then fall back to the acquisition date in the filename.
    parts = {part.lower() for part in path.parts}
    if 'prefire' in parts:
        return 'prefire'
    if 'postfire' in parts:
        return 'postfire'
    try:
        return classify_phase_from_acquisition(path)
    except ValueError:
        return None


def folder_has_tiffs(folder: Path) -> bool:
    # Confirm an extraction folder actually contains imagery before reusing it.
    return folder.exists() and any(iter_images(folder))


def extract_zip_to_phase_dir(zip_path: Path, root: Path) -> tuple[str, Path, str]:
    # Extract one ZIP into <root>/prefire/<zip-stem>/ or <root>/postfire/<zip-stem>/.
    # The source ZIP remains in place; only the derived TIFF folder is created.
    phase = classify_phase_from_acquisition(zip_path)
    extract_dir = root / phase / zip_path.stem

    if folder_has_tiffs(extract_dir):
        return phase, extract_dir, 'reused'

    if DRY_RUN:
        return phase, extract_dir, 'planned'

    if extract_dir.exists() and not folder_has_tiffs(extract_dir):
        # Remove a partial extraction from an earlier failed run.
        shutil.rmtree(extract_dir)

    work_dir = extract_dir.with_name(f'{extract_dir.name}_extracting')
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ['unzip', '-q', str(zip_path.resolve()), '-d', str(work_dir.resolve())],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        fallback_error = None
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(work_dir)
        except zipfile.BadZipFile as exc2:
            fallback_error = exc2
        except Exception as exc2:
            fallback_error = exc2

        if fallback_error is not None:
            shutil.rmtree(work_dir)
            message = result.stderr.strip() or result.stdout.strip()
            raise RuntimeError(
                f'unzip failed for {zip_path}: {message}\n'
                f'zipfile fallback also failed: {fallback_error}'
            )

    if not folder_has_tiffs(work_dir):
        shutil.rmtree(work_dir)
        raise RuntimeError(f'No TIFF files were extracted from {zip_path}')

    extract_dir.parent.mkdir(parents=True, exist_ok=True)
    work_dir.rename(extract_dir)
    return phase, extract_dir, 'extracted'


def extract_all_zips_to_phase_dirs(root: Path) -> dict[str, int]:
    # Find ZIPs recursively and extract each one into the phase folder determined by its date.
    counts = {'found': 0, 'planned': 0, 'extracted': 0, 'reused': 0, 'errors': 0}
    zip_files = sorted(iter_zips(root))
    counts['found'] = len(zip_files)

    for zip_path in zip_files:
        try:
            phase, extract_dir, status = extract_zip_to_phase_dir(zip_path, root)
            counts[status] += 1
            label = '[dry-run unzip]' if status == 'planned' else '[unzip]'
            print(f'{label} {zip_path} -> {extract_dir} ({phase}, {status})')
        except Exception as exc:
            counts['errors'] += 1
            print(f'[skip] Could not extract {zip_path}: {exc}')

    return counts


In [31]:
def make_spatial_ref_from_epsg(epsg_code: int) -> osr.SpatialReference:
    # Build an OSR spatial reference object from an EPSG integer.
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(epsg_code)
    srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
    return srs


def get_dataset_srs(ds: gdal.Dataset):
    # Prefer the coordinate reference system stored in the TIFF header.
    proj_wkt = ds.GetProjection()
    if proj_wkt:
        srs = osr.SpatialReference()
        srs.ImportFromWkt(proj_wkt)
        srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
        return srs

    # If no CRS exists in the file, use the fallback EPSG when configured.
    if FALLBACK_EPSG is not None:
        return make_spatial_ref_from_epsg(FALLBACK_EPSG)
    return None


def pixel_to_geo(gt, px: float, py: float) -> tuple[float, float]:
    # Convert pixel coordinates into map coordinates using the GDAL geotransform.
    x = gt[0] + px * gt[1] + py * gt[2]
    y = gt[3] + px * gt[4] + py * gt[5]
    return x, y


def estimate_gsd_meters(ds: gdal.Dataset, src_srs: osr.SpatialReference) -> float | None:
    # Estimate pixel size in meters by projecting one-pixel offsets to Web Mercator.
    gt = ds.GetGeoTransform(can_return_null=True)
    if gt is None:
        return None

    p0 = pixel_to_geo(gt, 0.0, 0.0)
    px = pixel_to_geo(gt, 1.0, 0.0)
    py = pixel_to_geo(gt, 0.0, 1.0)

    web_mercator = make_spatial_ref_from_epsg(3857)
    transform = osr.CoordinateTransformation(src_srs, web_mercator)
    p0m = transform.TransformPoint(*p0)
    pxm = transform.TransformPoint(*px)
    pym = transform.TransformPoint(*py)

    x_res = math.hypot(pxm[0] - p0m[0], pxm[1] - p0m[1])
    y_res = math.hypot(pym[0] - p0m[0], pym[1] - p0m[1])
    return float((x_res + y_res) / 2.0)


def zoom_for_full_resolution(gsd_m_per_px: float) -> int:
    # Convert ground sample distance to the nearest full-resolution Web Mercator zoom.
    if gsd_m_per_px <= 0:
        return 0
    zoom = math.ceil(math.log2(156543.03392804097 / gsd_m_per_px))
    return max(0, min(24, zoom))


def calculate_overview_levels(width: int, height: int) -> list[int]:
    # Choose powers-of-2 overviews so the smallest overview fits in one COG tile.
    max_dim = max(width, height)
    if max_dim <= COG_BLOCK_SIZE:
        return []
    num_levels = math.ceil(math.log2(max_dim / COG_BLOCK_SIZE))
    return [2 ** i for i in range(1, num_levels + 1)]


def check_is_cog(ds: gdal.Dataset) -> bool:
    # GDAL COGs expose IMAGE_STRUCTURE metadata LAYOUT=COG.
    return ds.GetMetadataItem('LAYOUT', 'IMAGE_STRUCTURE') == 'COG'


def find_root_for_image(image_path: Path, roots: list[Path]) -> Path | None:
    # Return the NAIP root that contains the image, if any.
    image_abs = image_path.resolve()
    for root in roots:
        root_abs = root.resolve()
        try:
            image_abs.relative_to(root_abs)
            return root_abs
        except ValueError:
            continue
    return None


def build_cog_creation_options(overview_count: int) -> list[str]:
    # Build creation options for a smaller but still lossless COG profile.
    creation_options = [
        'TILED=YES',
        f'COMPRESS={COG_COMPRESS}',
        f'LEVEL={COG_LEVEL}',
        f'PREDICTOR={COG_PREDICTOR}',
        f'BLOCKSIZE={COG_BLOCK_SIZE}',
        f'NUM_THREADS={COG_NUM_THREADS}',
        'BIGTIFF=IF_SAFER',
        f'OVERVIEW_RESAMPLING={OVERVIEW_RESAMPLING}',
        f'OVERVIEW_COMPRESS={OVERVIEW_COMPRESS}',
        f'OVERVIEW_PREDICTOR={OVERVIEW_PREDICTOR}',
        'COPY_SRC_OVERVIEWS=YES',
    ]

    if overview_count > 0:
        creation_options.append('OVERVIEWS=AUTO')
    else:
        creation_options.append('OVERVIEWS=NONE')

    return creation_options


def build_cog_from_source(src_image: Path, output_cog: Path, overview_levels: list[int]) -> bool:
    # Build one canonical COG from the original source image, preserving all bands.
    # Returns True when a file is written, or False if skipped due to existing output.
    if output_cog.exists() and not OVERWRITE_OUTPUT:
        return False

    output_cog.parent.mkdir(parents=True, exist_ok=True)
    creation_options = build_cog_creation_options(len(overview_levels))

    cog_ds = gdal.Translate(
        str(output_cog),
        str(src_image),
        format='COG',
        options=gdal.TranslateOptions(
            creationOptions=creation_options,
        ),
    )
    if cog_ds is None:
        raise RuntimeError(f'Failed to create COG: {output_cog}')

    cog_ds = None

    return True

## Cell 7: Helper Functions (Part 2)

This second helper cell focuses on COG-specific logic and path classification:

- `check_is_cog`
- `infer_fire_phase`
- `find_root_for_image`
- `build_cog_from_source`

## Cell 8: Discover Imagery and Build Canonical COG Plan

This cell recursively scans all configured roots for ZIP files and TIFF files.

If ZIPs are found, each archive is classified from the acquisition date in its filename and extracted into:

- `<NAIP root>/prefire/<zip filename without extension>/`
- `<NAIP root>/postfire/<zip filename without extension>/`

After extraction, the cell recursively scans for TIFFs and prepares a COG build plan. Output COGs are written without preserving the ZIP filename folder, so all products land directly in:

- `./<geojson-stem>/cog/prefire/`
- `./<geojson-stem>/cog/postfire/`


In [32]:
# Validate configured roots first so errors fail fast.
missing = [root for root in NAIP_ROOTS if not root.exists()]
if missing:
    raise FileNotFoundError(f'NAIP root(s) not found: {[str(path) for path in missing]}')

plans: list[CogPlan] = []
total_source_bytes = 0
zip_totals = {'found': 0, 'planned': 0, 'extracted': 0, 'reused': 0, 'errors': 0}

# ZIPs are handled first so newly extracted TIFFs are available to the recursive scan.
for root in NAIP_ROOTS:
    root_counts = extract_all_zips_to_phase_dirs(root)
    for key, value in root_counts.items():
        zip_totals[key] += value

# Gather all candidate TIFFs recursively from every configured NAIP root.
# iter_images() skips generated COG folders so reruns only process source imagery.
all_images = sorted(image for root in NAIP_ROOTS for image in iter_images(root))

if not all_images:
    print('No TIFFs were found to process. Check NAIP_ROOTS and ZIP extraction messages above.')

for image_path in all_images:
    size_bytes = image_path.stat().st_size
    total_source_bytes += size_bytes

    phase = infer_fire_phase(image_path)
    if phase not in {'prefire', 'postfire'}:
        print(f'[skip] Cannot infer prefire/postfire from source path or filename: {image_path}')
        continue

    ds = gdal.Open(str(image_path), gdal.GA_ReadOnly)
    if ds is None:
        print(f'[skip] Cannot open: {image_path}')
        continue

    src_srs = get_dataset_srs(ds)
    if src_srs is None:
        print(f'[skip] {image_path.name} has no CRS and FALLBACK_EPSG is disabled.')
        ds = None
        continue

    width, height = ds.RasterXSize, ds.RasterYSize
    band_count = ds.RasterCount
    source_is_cog = check_is_cog(ds)

    try:
        gsd_m = estimate_gsd_meters(ds, src_srs)
    except Exception as exc:
        print(f'[skip] {image_path.name} metadata error: {exc}')
        continue
    finally:
        ds = None

    native_zoom = zoom_for_full_resolution(gsd_m) if gsd_m else None
    overview_levels = calculate_overview_levels(width, height)

    # One COG per source TIFF, written directly to the GeoJSON-named phase folder.
    # This avoids carrying ZIP filename folders into the output path.
    output_cog = final_cog_dir_for_phase(phase) / f'{image_path.stem}.tif'

    plans.append(CogPlan(
        image_path=image_path,
        output_cog=output_cog,
        size_bytes=size_bytes,
        band_count=band_count,
        width=width,
        height=height,
        gsd_m=gsd_m,
        native_zoom=native_zoom,
        overview_levels=overview_levels,
        source_is_cog=source_is_cog,
        phase=phase,
    ))

# ---- Discovery summary for quick QA ------------------------------------------------
already_cog = sum(1 for plan in plans if plan.source_is_cog)
prefire_count = sum(1 for plan in plans if plan.phase == 'prefire')
postfire_count = sum(1 for plan in plans if plan.phase == 'postfire')
unknown_phase = sum(1 for plan in plans if plan.phase is None)

print('=== Discovery Summary ===')
print(f'Roots scanned:      {len(NAIP_ROOTS)}')
print(f'ZIPs found:         {zip_totals["found"]}')
if DRY_RUN:
    print(f'ZIPs planned unzip: {zip_totals["planned"]}')
else:
    print(f'ZIPs extracted:     {zip_totals["extracted"]}')
print(f'ZIPs reused:        {zip_totals["reused"]}')
print(f'ZIP errors:         {zip_totals["errors"]}')
print(f'Images discovered:  {len(plans)}')
print(f'Already COG:        {already_cog} / {len(plans)}')
print(f'Prefire:            {prefire_count}')
print(f'Postfire:           {postfire_count}')
print(f'Unknown phase:      {unknown_phase}')
print(f'Total source size:  {human_size(total_source_bytes)}')
print()

for plan in plans:
    gsd_text = f'{plan.gsd_m:.4f} m/px' if plan.gsd_m else 'unavailable'
    zoom_text = f'z={plan.native_zoom}' if plan.native_zoom is not None else 'unavailable'
    phase_text = plan.phase if plan.phase is not None else 'unknown'
    cog_flag = ' [already COG]' if plan.source_is_cog else ''

    if plan.native_zoom is not None and plan.overview_levels:
        ov_labels = [
            f'{factor}x (~z={max(0, plan.native_zoom - i)})'
            for i, factor in enumerate(plan.overview_levels, start=1)
        ]
    else:
        ov_labels = [f'{factor}x' for factor in plan.overview_levels]

    print(f'Image:        {plan.image_path}{cog_flag}')
    print(f'Bands:        {plan.band_count}  |  Size: {human_size(plan.size_bytes)}')
    print(f'Dimensions:   {plan.width} x {plan.height} px')
    print(f'Phase:        {phase_text}')
    print(f'GSD / Zoom:   {gsd_text}  |  {zoom_text}')
    print(f'Output COG:   {plan.output_cog}')
    print(f'Overviews:    {", ".join(ov_labels) if ov_labels else "none"}')
    print('---')


[unzip] /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/north_complex_2020/naip/m_3912001_se_10_060_20200710.ZIP -> downloads/north_complex_2020/naip/prefire/m_3912001_se_10_060_20200710 (prefire, reused)
[unzip] /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/north_complex_2020/naip/m_3912001_se_10_060_20220720.ZIP -> downloads/north_complex_2020/naip/postfire/m_3912001_se_10_060_20220720 (postfire, reused)
[unzip] /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/north_complex_2020/naip/m_3912001_sw_10_060_20200710.ZIP -> downloads/north_complex_2020/naip/prefire/m_3912001_sw_10_060_20200710 (prefire, reused)
[unzip] /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/north_complex_2020/naip/m_3912001_sw_10_060_20220720.ZIP -> downloads/north_complex_2020/naip/postfire/m_3912001_sw_10_060_20220720 (postfire, reused)
[unzip] /Users/maples/GitHub/EarthExplorer_API/notebooks/downloads/north_complex_2020/naip/m_3912002_se_10_060_20200701.ZIP -> downl

Warning 1: m_3912105_se_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3912113_ne_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3912114_ne_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3912114_nw_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: m_3912114_se_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as

## Cell 10: Build Canonical COGs (Keep All Source Bands)

For each source image in the plan this cell writes exactly one COG that preserves all original source bands.

Output paths are:

- `./<geojson-stem>/cog/prefire/<image>.tif`
- `./<geojson-stem>/cog/postfire/<image>.tif`

Behavior:

- Source TIFF with `N` bands -> output COG with the same `N` bands.
- RGB and IRG visualization choices are handled later by VRTs, not by duplicating raster storage.
- Existing COGs are skipped unless `OVERWRITE_OUTPUT=True`.


In [33]:
cogs_built: list[Path] = []
images_built = 0
images_errored = 0

for p in plans:
    if DRY_RUN:
        ov_str = str(p.overview_levels) if p.overview_levels else '[]'
        print(f'[dry-run] {p.image_path.name}')
        print(f'  source bands: {p.band_count}')
        print(f'  output COG:   {p.output_cog}')
        print(f'  overviews:    {ov_str}')
        cogs_built.append(p.output_cog)
        continue

    try:
        written = build_cog_from_source(p.image_path, p.output_cog, p.overview_levels)
        cogs_built.append(p.output_cog)
        images_built += 1
        status = 'written' if written else 'skipped (exists)'
        print(f'[done] {p.image_path.name} -> {p.output_cog.name} ({status})')
    except Exception as exc:
        images_errored += 1
        print(f'[error] {p.image_path.name}: {exc}')

print()
print('=== Canonical COG Build Summary ===')
if DRY_RUN:
    print(f'Mode: DRY-RUN  |  Images planned: {len(plans)}')
    print('Set DRY_RUN=False and re-run Cells 8, 10, 12, and 14 to write output.')
else:
    print(f'Mode: WRITE  |  Images built: {images_built}  |  Errors: {images_errored}')

[done] m_3912001_se_10_060_20220720.tif -> m_3912001_se_10_060_20220720.tif (written)
[done] m_3912001_sw_10_060_20220720.tif -> m_3912001_sw_10_060_20220720.tif (written)
[done] m_3912002_se_10_060_20220720.tif -> m_3912002_se_10_060_20220720.tif (written)
[done] m_3912002_sw_10_060_20220720.tif -> m_3912002_sw_10_060_20220720.tif (written)
[done] m_3912009_ne_10_060_20220720.tif -> m_3912009_ne_10_060_20220720.tif (written)
[done] m_3912009_nw_10_060_20220720.tif -> m_3912009_nw_10_060_20220720.tif (written)
[done] m_3912009_se_10_060_20220720.tif -> m_3912009_se_10_060_20220720.tif (written)
[done] m_3912009_sw_10_060_20220720.tif -> m_3912009_sw_10_060_20220720.tif (written)
[done] m_3912010_ne_10_060_20220720.tif -> m_3912010_ne_10_060_20220720.tif (written)
[done] m_3912010_nw_10_060_20220720.tif -> m_3912010_nw_10_060_20220720.tif (written)
[done] m_3912011_nw_10_060_20220720.tif -> m_3912011_nw_10_060_20220720.tif (written)


Warning 1: m_3912105_se_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3912105_se_10_060_20220719.tif -> m_3912105_se_10_060_20220719.tif (written)
[done] m_3912106_se_10_060_20220719.tif -> m_3912106_se_10_060_20220719.tif (written)
[done] m_3912106_sw_10_060_20220719.tif -> m_3912106_sw_10_060_20220719.tif (written)
[done] m_3912107_sw_10_060_20220719.tif -> m_3912107_sw_10_060_20220719.tif (written)


Warning 1: m_3912113_ne_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3912113_ne_10_060_20220719.tif -> m_3912113_ne_10_060_20220719.tif (written)
[done] m_3912113_se_10_060_20220714.tif -> m_3912113_se_10_060_20220714.tif (written)


...

Warning 1: m_3912114_se_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3912114_se_10_060_20220719.tif -> m_3912114_se_10_060_20220719.tif (written)


Warning 1: m_3912114_sw_10_060_20220719.tif: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[done] m_3912114_sw_10_060_20220719.tif -> m_3912114_sw_10_060_20220719.tif (written)
[done] m_3912115_ne_10_060_20220719.tif -> m_3912115_ne_10_060_20220719.tif (written)
[done] m_3912115_nw_10_060_20220719.tif -> m_3912115_nw_10_060_20220719.tif (written)
[done] m_3912115_se_10_060_20220719.tif -> m_3912115_se_10_060_20220719.tif (written)
[done] m_3912115_sw_10_060_20220719.tif -> m_3912115_sw_10_060_20220719.tif (written)
[done] m_3912116_ne_10_060_20220719.tif -> m_3912116_ne_10_060_20220719.tif (written)
[done] m_3912116_nw_10_060_20220719.tif -> m_3912116_nw_10_060_20220719.tif (written)
[done] m_3912116_se_10_060_20220719.tif -> m_3912116_se_10_060_20220719.tif (written)
[done] m_3912116_sw_10_060_20220719.tif -> m_3912116_sw_10_060_20220719.tif (written)
[done] m_3912121_ne_10_060_20220714.tif -> m_3912121_ne_10_060_20220714.tif (written)


KeyboardInterrupt: 

## Cell 12: Optional Visualization VRTs from Canonical COGs

If `BUILD_VIZ_VRTS=True`, this cell builds two aggregate VRT mosaics from canonical COGs:

- `rgb_all.vrt` using bands `1,2,3`
- `irg_all.vrt` using bands `4,1,2`

This keeps storage efficient (single canonical COGs) while still supporting common visualization styles.

In [ ]:
def build_visualization_vrt(cog_list: list[Path], output_vrt: Path, bands: tuple[int, int, int], label: str) -> None:
    # Build one aggregate VRT with a specific visualization band recipe.
    if not cog_list:
        print(f'[skip] No source COGs available for {label}.')
        return

    output_vrt.parent.mkdir(parents=True, exist_ok=True)
    ds = gdal.BuildVRT(
        str(output_vrt),
        [str(p) for p in cog_list],
        options=gdal.BuildVRTOptions(
            resolution='highest',
            allowProjectionDifference=True,
            bandList=list(bands),
        ),
    )
    if ds is None:
        raise RuntimeError(f'gdal.BuildVRT failed for: {output_vrt}')
    ds.FlushCache()
    ds = None
    print(f'[done] {label}: {output_vrt}  ({len(cog_list)} source COGs)')


if not BUILD_VIZ_VRTS:
    print('BUILD_VIZ_VRTS=False -> skipping aggregate RGB/IRG VRT creation.')
elif DRY_RUN:
    print('[dry-run] Aggregate VRT targets:')
    print(f'  RGB: {RGB_AGGREGATE_VRT} using bands {RGB_BANDS}')
    print(f'  IRG: {IRG_AGGREGATE_VRT} using bands {IRG_BANDS}')
    print('Set DRY_RUN=False to write VRT files.')
else:
    existing_cogs = [p for p in cogs_built if p.exists()]
    print(f'Canonical COGs found on disk: {len(existing_cogs)} / {len(cogs_built)}')

    # Only build IRG if there are enough bands in every source image.
    min_bands = min((p.band_count for p in plans), default=0)
    build_visualization_vrt(existing_cogs, RGB_AGGREGATE_VRT, RGB_BANDS, 'RGB mosaic')
    if min_bands >= 4:
        build_visualization_vrt(existing_cogs, IRG_AGGREGATE_VRT, IRG_BANDS, 'IRG mosaic')
    else:
        print('[skip] IRG VRT requires band 4; at least one source image has < 4 bands.')

## Recommended Run Order

1. Run the GDAL environment preflight cell.
2. Edit Cell 4 configuration:
   - Set `AOI_GEOJSON` to the fire perimeter GeoJSON you are preparing.
   - Set `NAIP_ROOTS` to the local folders containing NAIP ZIPs or TIFFs.
   - Confirm `PREFIRE_YEAR = 2020`.
3. Run the helper cells.
4. Run Cell 8 to recursively find ZIPs/TIFFs, unzip ZIPs into `prefire` or `postfire`, and build the COG plan.
5. Run Cell 10 to write COGs to `./<geojson-stem>/cog/prefire/` and `./<geojson-stem>/cog/postfire/`.
6. Optionally run the VRT cell to create aggregate RGB and IRG VRTs.


In [ ]:
# No consolidation is needed.
# Cell 8 plans final output paths directly under OUTPUT_BASE_DIR, and Cell 10
# writes each COG straight to ./<geojson-stem>/cog/prefire or /postfire.

planned_outputs = [plan.output_cog for plan in plans]
prefire_outputs = [path for path in planned_outputs if 'prefire' in {part.lower() for part in path.parts}]
postfire_outputs = [path for path in planned_outputs if 'postfire' in {part.lower() for part in path.parts}]

print('=== Consolidation Not Required ===')
print('Cell 10 already writes COGs directly to final local phase folders.')
print(f'Output base dir : {OUTPUT_BASE_DIR.resolve()}')
print(f'Prefire COGs    : {len(prefire_outputs)} planned')
print(f'Postfire COGs   : {len(postfire_outputs)} planned')
print('No files were moved.')


## Cell 13 Notes: Final Output Behavior

- Final COG outputs go to `./<geojson-stem>/cog/prefire/` or `./<geojson-stem>/cog/postfire/`.
- ZIPs are extracted into local phase folders under each configured NAIP root before COG creation.
- Generated COG folders are skipped during recursive source discovery so reruns do not process prior outputs.
- No server-specific write logic is used.


## Remote COG VRT: Stanford SDR

Build RGB and IRG VRT files using a single remote COG hosted on the Stanford Spatial Data Repository.
The `/vsicurl/` prefix tells GDAL to stream the file over HTTPS without downloading it first.


In [ ]:
from pathlib import Path
from osgeo import gdal

gdal.UseExceptions()

# Remote COG on Stanford SDR — GDAL streams it via /vsicurl/ without downloading.
REMOTE_COG_URL = "https://stacks.stanford.edu/file/kv186fx7335/m_3611854_se_11_060_20200726.tif"
VSICURL_PATH = f"/vsicurl/{REMOTE_COG_URL}"

# Output VRTs will sit next to the notebook.
OUT_DIR = Path(".")
RGB_VRT = OUT_DIR / "m_3611854_se_11_060_20200726_rgb.vrt"
IRG_VRT = OUT_DIR / "m_3611854_se_11_060_20200726_irg.vrt"

# ── RGB (bands 1, 2, 3 → Red, Green, Blue) ───────────────────────────────────
ds_rgb = gdal.BuildVRT(
    str(RGB_VRT),
    [VSICURL_PATH],
    options=gdal.BuildVRTOptions(
        resolution="highest",
        bandList=[1, 2, 3],   # Natural color: R, G, B from a 4-band NAIP COG
    ),
)
if ds_rgb is None:
    raise RuntimeError(f"gdal.BuildVRT failed for: {RGB_VRT}")
ds_rgb.FlushCache()
ds_rgb = None
print(f"[done] RGB VRT: {RGB_VRT.resolve()}")

# ── IRG (bands 4, 1, 2 → NIR, Red, Green — CIR false colour) ─────────────────
ds_irg = gdal.BuildVRT(
    str(IRG_VRT),
    [VSICURL_PATH],
    options=gdal.BuildVRTOptions(
        resolution="highest",
        bandList=[4, 1, 2],   # CIR false colour: NIR, R, G
    ),
)
if ds_irg is None:
    raise RuntimeError(f"gdal.BuildVRT failed for: {IRG_VRT}")
ds_irg.FlushCache()
ds_irg = None
print(f"[done] IRG VRT: {IRG_VRT.resolve()}")


In [ ]:
# Re-request the remote COG URL and print detailed errors, then test a download.
from pathlib import Path
import traceback
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from osgeo import gdal

# Reuse the URL and /vsicurl path from the previous cell when available.
url = globals().get(
    "REMOTE_COG_URL",
    "https://stacks.stanford.edu/file/kv186fx7335/m_3611854_se_11_060_20200726.tif",
)
vsicurl_path = globals().get("VSICURL_PATH", f"/vsicurl/{url}")

probe_dir = Path("downloads")
probe_dir.mkdir(parents=True, exist_ok=True)
probe_path = probe_dir / "m_3611854_se_11_060_20200726.download_probe.tif"
debug_vrt = probe_dir / "m_3611854_se_11_060_20200726.debug_rgb.vrt"

def print_headers(resp):
    print(f"HTTP status: {getattr(resp, 'status', 'unknown')}")
    print("Selected headers:")
    for h in ["Content-Type", "Content-Length", "Accept-Ranges", "ETag", "Last-Modified"]:
        print(f"  {h}: {resp.headers.get(h)}")

def try_request(method: str, byte_range: str | None = None):
    print(f"\n=== {method} request ===")
    headers = {"User-Agent": "naip-cog-builder-diagnostic/1.0"}
    if byte_range:
        headers["Range"] = byte_range
        print(f"Range: {byte_range}")

    req = Request(url, method=method, headers=headers)
    try:
        with urlopen(req, timeout=60) as resp:
            print_headers(resp)
            chunk = resp.read(1024)
            print(f"Read {len(chunk)} bytes from response body.")
            return True
    except HTTPError as e:
        print(f"HTTPError: {e}")
        print(f"Status: {e.code}")
        try:
            body = e.read(4096)
            if body:
                print("Error body (first 4KB):")
                print(body.decode("utf-8", errors="replace"))
        except Exception:
            print("Unable to read HTTP error body.")
        traceback.print_exc(limit=2)
    except URLError as e:
        print(f"URLError: {e}")
        traceback.print_exc(limit=2)
    except Exception as e:
        print(f"Unexpected error: {type(e).__name__}: {e}")
        traceback.print_exc()
    return False

# 1) HEAD check
_ = try_request("HEAD")

# 2) GET with a small byte range to verify partial-download capability
_ = try_request("GET", byte_range="bytes=0-1048575")

# 3) Download probe: write up to 25 MiB to disk to validate streaming download path.
print("\n=== Download probe (up to 25 MiB) ===")
max_bytes = 25 * 1024 * 1024
written = 0
req = Request(url, method="GET", headers={"User-Agent": "naip-cog-builder-diagnostic/1.0"})
try:
    with urlopen(req, timeout=120) as resp, open(probe_path, "wb") as f:
        while True:
            block = resp.read(1024 * 1024)  # 1 MiB chunks
            if not block:
                break
            remaining = max_bytes - written
            if remaining <= 0:
                break
            chunk = block[:remaining]
            f.write(chunk)
            written += len(chunk)
            if written >= max_bytes:
                break
    print(f"Download probe wrote {written} bytes to: {probe_path.resolve()}")
except HTTPError as e:
    print(f"HTTPError during download probe: {e} (status={e.code})")
    try:
        print(e.read(4096).decode("utf-8", errors="replace"))
    except Exception:
        pass
    traceback.print_exc(limit=2)
except URLError as e:
    print(f"URLError during download probe: {e}")
    traceback.print_exc(limit=2)
except Exception as e:
    print(f"Unexpected error during download probe: {type(e).__name__}: {e}")
    traceback.print_exc()

# 4) GDAL /vsicurl + BuildVRT diagnostics to expose GDAL-side errors.
print("\n=== GDAL /vsicurl diagnostics ===")
gdal_errors = []

def _gdal_error_handler(err_class, err_no, err_msg):
    gdal_errors.append((err_class, err_no, err_msg))

gdal.PushErrorHandler(_gdal_error_handler)
try:
    gdal.ErrorReset()
    ds = gdal.OpenEx(vsicurl_path, gdal.OF_RASTER)
    if ds is None:
        print("gdal.OpenEx returned None")
        print(f"GDAL last error: [{gdal.GetLastErrorNo()}] {gdal.GetLastErrorMsg()}")
    else:
        print(f"Opened via GDAL: {vsicurl_path}")
        print(f"Raster size: {ds.RasterXSize} x {ds.RasterYSize} | bands={ds.RasterCount}")
        ds = None

    gdal.ErrorReset()
    vrt_ds = gdal.BuildVRT(
        str(debug_vrt),
        [vsicurl_path],
        options=gdal.BuildVRTOptions(
            resolution="highest",
            bandList=[1, 2, 3],
        ),
    )
    if vrt_ds is None:
        print("gdal.BuildVRT returned None")
        print(f"GDAL last error: [{gdal.GetLastErrorNo()}] {gdal.GetLastErrorMsg()}")
    else:
        vrt_ds.FlushCache()
        vrt_ds = None
        print(f"gdal.BuildVRT succeeded: {debug_vrt.resolve()}")
except Exception as e:
    print(f"Exception from GDAL call: {type(e).__name__}: {e}")
    traceback.print_exc(limit=2)
finally:
    gdal.PopErrorHandler()

if gdal_errors:
    print("Captured GDAL error/warning messages:")
    for cls, num, msg in gdal_errors:
        print(f"  class={cls} code={num} msg={msg}")
else:
    print("No GDAL errors/warnings captured.")